# Registering a New Patch-Clamp `Experiment` in the Utah Organoids Pipeline

### **Overview**

This notebook guides you through the process of registering a new patch-clamp experiment for analysis in the Utah Organoids pipeline.
- An `EphysExperimentsForAnalysis` entry registers the experiment and its data location.
- A `CurrentStepTimeParams` entry defines the analysis time window for current-step protocols.

By the end of this notebook, you will have:
- Registered a new experiment in `EphysExperimentsForAnalysis`
- Defined timing parameters in `CurrentStepTimeParams`
- Triggered automatic processing by workers

**_Note:_**

- Before running this notebook, ensure ABF files and experiment metadata Excel are uploaded to S3.
- Data must follow the [file structure guidelines](../docs/installation_and_configuration/DATA_ORGANIZATION.md).
- The `experiment` identifier must match the folder name in S3 (e.g., `2020-08-28`).

### **Key Steps**

- **Setup**

- **Step 1: Define Experiment Details**

- **Step 2: Define Timing Parameters**

- **Step 3: Insert into Database**

#### **Setup**

First, import the necessary packages for the data pipeline and essential schemas.

In [ ]:
## Ensure the correct working directory

import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
import datajoint as dj

In [ ]:
from workflow.pipeline import patch_clamp

#### **Step 1: Define Experiment Details**

Each patch-clamp experiment must be registered in `EphysExperimentsForAnalysis` before it can be processed. Check existing registered experiments:

In [ ]:
patch_clamp.EphysExperimentsForAnalysis()

Define the experiment to register:

In [ ]:
# Update the following information to match your specific needs
experiment = "2020-08-28"  # Must match folder name in S3
project = "utah_organoids"
directory = f"patch_clamp/{experiment}/"  # Path relative to S3 raw data root

Build the experiment entry:

In [ ]:
experiment_entry = dict(
    experiment=experiment,
    project=project,
    use="Yes",
    directory=directory,
)

experiment_entry

#### **Step 2: Define Timing Parameters**

The `CurrentStepTimeParams` table defines the analysis time window for current-step protocols. Check existing timing parameters:

In [ ]:
patch_clamp.CurrentStepTimeParams()

#### 1. Define time window

- `istep_start`: When the current injection begins (seconds)
- `istep_duration`: How long to analyze (seconds)

The default values (0.55s start, 1.0s duration) are standard for most protocols.

**Note:** These timing values were determined by analyzing the ABF epoch table structure during baseline testing. The epoch table in ABF files defines when stimuli are applied — for the 2020-08-28 experiment, the current step begins at 0.55s. If your experiment uses a different protocol, inspect your ABF files to determine the correct timing parameters.

In [ ]:
# Update the following information to match your specific needs
istep_start = 0.55  # Current injection start time (seconds)
istep_duration = 1.0  # Analysis window duration (seconds)

#### 2. Compute derived values

In [ ]:
# Computed values (do not modify)
istep_end_1s = istep_start + 1.0  # End of first second (used for analysis)
istep_end = istep_start + istep_duration  # Actual end time

# Ensure istep_end_1s doesn't exceed actual end
if istep_end < istep_end_1s:
    istep_end_1s = istep_end

Build the timing parameters entry:

In [ ]:
timing_entry = dict(
    experiment=experiment,
    istep_start=istep_start,
    istep_end_1s=istep_end_1s,
    istep_end=istep_end,
    istep_duration=istep_duration,
)

timing_entry

#### **Step 3: Insert into Database**

Now, insert the experiment registration and timing parameters into the database.

In [ ]:
patch_clamp.EphysExperimentsForAnalysis.insert1(experiment_entry, skip_duplicates=True)

In [ ]:
patch_clamp.CurrentStepTimeParams.insert1(timing_entry, skip_duplicates=True)

Confirm experiment registration:

In [ ]:
patch_clamp.EphysExperimentsForAnalysis() & {"experiment": experiment}

In [ ]:
patch_clamp.CurrentStepTimeParams() & {"experiment": experiment}

### **Next Steps**

Now that the experiment is registered, you can:
- Monitor processing progress in the Dashboard under `Patch-Clamp` section
- Explore results using `EXPLORE` notebooks to query the `patch_clamp` schema